# Conversion to ib97sx failed


# This uses Sandy test results from ib970 in wsl directory ib97sx
***folder_path="/home/ratlabs/JL_2/data/ib97sx" [WSL]***
 *REPO ____ "JL_2/data/ib97sx"*
1. Find results in C:\Users\bhuns\OneDrive\***ib_97*** not present in ***"\\wsl.localhost\Ubuntu-20.04\home\ratlabs\JL_2\data\ib97sx"*** copy them and
2. paste them into the WSL folder
3. and select them
4.  then Use "F2 ExplorerCopy rename to remove "z_" prefix of the first one
5.  Then hit ***cntrl enter*** to replace the rest
6.  Then reset the workbook and run all cells
7.  Go the graphs

# col_wise_import ib970 individual test results to pickle [df_mf_ib97sx_nn_s]
1. ib970 exports test results to onedrive 1 csv file[uft-8] per test to ib97sx
2. manual copy file to repo/data ib97sx
3. import csv ib97sx files to unfiltered to [df_ib97sx_raw_uf]
4. Filter on the basis of ID  to [df_ib97sx_raw]
5. Strips off col numbers [df_ib97sx_nn]
6. Eliminate col duplicates by clean the numberless lables of hidden characters [df_ib97sx_nn]
7. Create media data cols [df_m_ib97sx_nn]
8. Computes media cols from data [df_mf_ib97sx_nn]
9. Removes timestamp duplicates to get [df_mf_ib97sx_nn]
10. Sorts via timestamp to get [df_mf_ib97sx_nn_s]
11. Writes to Pickle [df_mf_ib97sx_nn_s]
12. misc functions to be removed

Similarity of 

# Set up

In [121]:
import sys
print(sys.executable)
print("note: THIS IS THE DIRECTORY PYTHON IS WORKING IN.")

/home/ratlabs/JL_2/.venv/bin/python3
note: THIS IS THE DIRECTORY PYTHON IS WORKING IN.


In [122]:
# Imports required for Loading, sorting .csx files to create specific data sets ie mrn inbody readings. 
%run ./sys_funcs.py              # loads all the def functions in sys_funcs.py into memory
#import sys_funcs                 # gives access to these def function digitalform that are in memory
from pathlib import Path
import csv
import pandas as pd
import numpy as np
# import tkinter as tk
import pickle
from pathlib import Path
import csv
import os
import sys
from datetime import datetime
from datetime import time
from sys_funcs import read_csv_to_array
from sys_funcs import clean_wsl_path
from sys_funcs import array_to_dt_row_dict
from sys_funcs import make_blnk_update_row_dict
from sys_funcs import transpose_csv_to_col_dict
#from sys_funcs import update_values_with_config, get_update_result
from sys_funcs import transfer_updates
from sys_funcs import get_dtv_range
from sys_funcs import universal_import
from sys_funcs import parse_inbody_timestamp
from sys_funcs import build_lut
from sys_funcs import extract_a_column_as_df
from sys_funcs import extract_multicolumns_as_df
from sys_funcs import validate_and_sort_timestamps
from sys_funcs import extract_and_filter_by_time_window
from sys_funcs import read_file_dual_path
from sys_funcs import write_file_dual_path
from sys_funcs import asc_to_csv_cnv
from collections.abc import Mapping
import re
#from sys_funcs import 

In [123]:
# set print rows  This worksheet sets maximum # of rows printed
pd.set_option('display.max_rows', 1000)  # Adjust the number of rows to display
# pd.reset_option('display.max_rows')  
print('print set to 1000 rows max' )

print set to 1000 rows max


In [124]:
print("NOTE: timestamp = Test Date / Time does not work  use computed time stamp")


NOTE: timestamp = Test Date / Time does not work  use computed time stamp


# Def functions called in data importing & refinment.

In [125]:
# def concert df to dct
def df_dct(df):
    return {col: df[col] for col in df.columns}


In [126]:
media_lst = [
    "timestamp",
    "dtv",
    "ib_id",
    "cls",
    "cmmnts"
]

In [127]:
# 3rd version def drop_duplicates_by_test_time(df, keep='first', log=True):
def drop_duplicates_by_test_time(df, keep='first', log=True):
    """
    Removes duplicate rows based on the 'Test Date / Time' column.
    Keeps only the first (or last) occurrence.
    """
    df = df.copy()

    # Identify duplicate timestamps (beyond the one we keep)
    dupes = (
        df.loc[df.duplicated(subset=['Test Date / Time'], keep=keep), 'Test Date / Time']
        .astype(str)
        .values
        .tolist()
    )

    # Drop duplicate rows
    df = df.drop_duplicates(subset=['Test Date / Time'], keep=keep)

    # Optional logging
    if log and dupes:
        print("Removed duplicate rows for timestamps:", dupes)

    return df


In [128]:
# def strip_numbers_from_columns(df):
import re

def strip_numbers_from_columns(df):
    """
    Removes leading/trailing numbers and any leftover separators
    so that cases like '1.0ID' become 'ID'.
    """
    df = df.copy()
    new_cols = {}

    for col in df.columns:
        cleaned = col

        # Remove leading numbers + separators
        cleaned = re.sub(r'^\d+[\s\-\_\.:]*', '', cleaned)

        # Remove trailing numbers + separators
        cleaned = re.sub(r'[\s\-\_\.:]*\d+$', '', cleaned)

        # Remove leftover leading/trailing punctuation (.,-_:) after number removal
        cleaned = re.sub(r'^[\.\-\_\:]+', '', cleaned)
        cleaned = re.sub(r'[\.\-\_\:]+$', '', cleaned)

        new_cols[col] = cleaned

    return df.rename(columns=new_cols)


In [129]:
# Use drop duplicate function If duplicates are found
def drop_duplicate_columns(df, keep='first', log=True):
    """
    Removes duplicate column names from a DataFrame, keeping only the first
    (or last) occurrence. Useful after column-cleaning steps that may cause
    collisions. good i'm moving this around because I want to go ahead and do the I'm talking too

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame.
    keep : {'first', 'last'}, default 'first'
        Which duplicate to keep.
    log : bool, default True
        Whether to print which columns were removed.

    Returns
    -------
    pandas.DataFrame
        DataFrame with duplicate columns removed.
    """
    df = df.copy()

    # Identify duplicates beyond the one we keep
    dupes = df.columns[df.columns.duplicated(keep=keep)].tolist()

    # Drop them
    df = df.loc[:, ~df.columns.duplicated(keep=keep)]

    # Optional logging
    if log and dupes:
        print("Removed duplicate columns:", dupes)

    return df


In [130]:
# def prepend_empty_columns(df, col_list):
def prepend_empty_columns(df, col_list):
    """
    Prepend empty columns (from col_list) to the front of df.
    Returns a new DataFrame.
    """
    import pandas as pd

    # Create empty columns with same row count
    empty_df = pd.DataFrame(
        {col: [None] * len(df) for col in col_list}
    )

    # Prepend them
    return pd.concat([empty_df, df], axis=1)


In [131]:
# revised def fill_ib_media_cols(df):
def fill_ib_media_cols(df):
    df = df.copy()

    # --- 1. Clean and parse timestamp --------------------------
    def fix_ts(x):
        if pd.isna(x):
            return np.nan
        # Convert float → int safely
        try:
            x_int = int(float(x))
        except:
            return np.nan
        # Zero‑pad to 14 digits (YYYYMMDDHHMMSS)
        s = str(x_int).zfill(14)
        return pd.to_datetime(s, format="%Y%m%d%H%M%S", errors="coerce")

    df['timestamp'] = df['Test Date / Time'].apply(fix_ts)

    # --- 2. dtv ------------------------------------------------
    origin = pd.Timestamp("1900-01-01")
    df['dtv'] = (df['timestamp'] - origin).dt.days

    # --- 3. ib_id ----------------------------------------------
    def classify_ib_id(ts):
        if pd.isna(ts):
            return np.nan
        hour = ts.hour
        return "mrn" if 3 <= hour <= 12 else "eve"

    df['ib_id'] = df['timestamp'].apply(classify_ib_id)

    # --- 4–5. cls, cmmnts --------------------------------------
    df['cls'] = np.nan
    df['cmmnts'] = np.nan

    return df


In [132]:
# Sort the rows by timestamp
def sort_by_timestamp(df):
    """
    Sorts an InBody dataframe by the 'timestamp' column
    in ascending chronological order.
    """
    df = df.copy()
    df = df.sort_values(by='timestamp', ascending=True)
    df = df.reset_index(drop=True)
    return df


In [133]:
# A function to combine frames of ib97sx with ib97sx and sorting into one data frame. N
# this will be used on a column by column basis for a list of columns.
def combine_weight_frames(df_a, df_b, ts_col="timestamp", wt_a="2. wt", wt_b="4. wt"):
    """
    Combines two dataframes with different weight column names into a single
    dataframe with columns: timestamp, wt, sorted by timestamp.

    Parameters
    ----------
    df_a : pd.DataFrame
        First dataframe containing a timestamp column and a weight column.
    df_b : pd.DataFrame
        Second dataframe containing a timestamp column and a weight column.
    ts_col : str, optional
        Name of the timestamp column (default 'timestamp').
    wt_a : str, optional
        Weight column name in df_a (default '2. wt').
    wt_b : str, optional
        Weight column name in df_b (default '4. wt').

    Returns
    -------
    pd.DataFrame
        Combined dataframe with columns: timestamp, wt, sorted by timestamp.
    """

    # Normalize df_a
    df_a_norm = df_a[[ts_col, wt_a]].rename(columns={wt_a: "wt"})

    # Normalize df_b
    df_b_norm = df_b[[ts_col, wt_b]].rename(columns={wt_b: "wt"})

    # Stack them vertically
    df_combined = pd.concat([df_a_norm, df_b_norm], ignore_index=True)

    # Sort by timestamp
    df_combined = df_combined.sort_values(by=ts_col).reset_index(drop=True)

    return df_combined


In [134]:
# remove duplicates on the basis of timestamp
def remove_ib_duplicates(df, subset_cols=None):
    """
    Removes duplicate InBody rows based on key identifying columns.
    Default behavior: remove duplicates based on ['ID', 'timestamp'].
    """
    df = df.copy()

    # Default duplicate definition
    if subset_cols is None:
        subset_cols = ['timestamp']
        # subset_cols = ['ID', 'timestamp']
    # Remove duplicates, keeping the first occurrence
    df = df.drop_duplicates(subset=subset_cols, keep='first')

    # Reset index for cleanliness
    df = df.reset_index(drop=True)

    return df


In [135]:
# veirfy if rows exixt in master_timestamps(df_master, df_new, ts_col="timestamp"):
def filter_new_rows_by_master_timestamps(df_master, df_new, ts_col="timestamp"):
    """
    Filters df_new so that only rows whose timestamps appear in df_master remain.

    Parameters
    ----------
    df_master : pd.DataFrame
        The master dataframe containing valid timestamps.
    df_new : pd.DataFrame
        The new dataframe to be filtered.
    ts_col : str, optional
        The name of the timestamp column (default is 'timestamp').

    Returns
    -------
    pd.DataFrame
        A filtered version of df_new containing only rows whose timestamps
        exist in df_master.
    """

    # Extract the set of valid timestamps from the master dataframe
    valid_timestamps = set(df_master[ts_col])

    # Filter df_new to keep only rows with timestamps in the master set
    df_filtered = df_new[df_new[ts_col].isin(valid_timestamps)].copy()

    return df_filtered


In [136]:
# def keep_only_new_timestamps(df_master, df_new, ts_col="timestamp")
def keep_only_new_timestamps(df_master, df_new, ts_col="timestamp"):
    """
    Returns only the rows in df_new whose timestamps do NOT exist in df_master.

    Parameters
    ----------
    df_master : pd.DataFrame
        The master dataframe containing timestamps already ingested.
    df_new : pd.DataFrame
        The new dataframe to be filtered.
    ts_col : str, optional
        The name of the timestamp column (default is 'timestamp').

    Returns
    -------
    pd.DataFrame
        A filtered version of df_new containing only rows with timestamps
        NOT present in df_master.
    """

    # Extract the set of timestamps already in the master
    existing_ts = set(df_master[ts_col])

    # Keep only rows whose timestamp is NOT in the master
    df_filtered = df_new[~df_new[ts_col].isin(existing_ts)].copy()

    return df_filtered


In [137]:
# def append_rows_with_master_schema(master_df, adder_df):
import pandas as pd
import numpy as np

def append_rows_with_master_schema(master_df, adder_df):
    """
    Appends rows from adder_df into master_df while enforcing the master_df schema.

    For each row in adder_df:
      - Columns that exist in adder_df are copied.
      - Columns missing from adder_df are filled with NaN.
      - All master_df columns are preserved in order.

    Parameters
    ----------
    master_df : pd.DataFrame
        The master dataframe with the full schema.
    adder_df : pd.DataFrame
        The dataframe containing rows to append (subset of master columns).

    Returns
    -------
    pd.DataFrame
        Updated master_df with new rows appended.
    """

    # Reindex adder_df to match master_df columns, filling missing columns with NaN
    adder_aligned = adder_df.reindex(columns=master_df.columns)

    # Append and return
    return pd.concat([master_df, adder_aligned], ignore_index=True)


In [138]:
# def filter_by_value(df, column, value):
def filter_by_value(df, column, value):
    """
    Returns a filtered DataFrame containing only rows where df[column] == value.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to filter.
    column : str
        The column name to filter on.
    value : any
        The value that the column must match.

    Returns
    -------
    pd.DataFrame
        A filtered dataframe containing only matching rows.
    """
    return df[df[column] == value].copy()


In [139]:
# OLD VERSION def write_df_to_pickle(df, filename):
def write_df_to_pickle(df, filename):
    """
    Writes a DataFrame to a pickle file.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to save.
    filename : str
        The pickle filename, e.g. 'mydata.pkl'.
    """
    df.to_pickle(filename)

# usage 
# write_df_to_pickle(df, "df.pkl")


In [140]:
# OLD VERSION def load_df_from_pickle(filename):

def load_df_from_pickle(filename):
    """
    Loads a DataFrame from a pickle file.

    Parameters
    ----------
    filename : str
        Path to the pickle file.

    Returns
    -------
    pd.DataFrame
    """
    return pd.read_pickle(filename)

    # usage 
    # df = load_df_from_pickle("df.pkl")



In [141]:
# def scale_mean_to_one(series):
def scale_mean_to_one(series):
    """Scale a Pandas Series so that its mean becomes 1."""
    mean_val = series.mean()
    return series / mean_val


In [142]:
# def plot_column(df, col_name):
import matplotlib.pyplot as plt

def plot_column(df, col_name):
    """
    Plot a single column from a dataframe.
    
    Parameters
    ----------
    df : pandas.DataFrame
        The dataframe containing the column.
    col_name : str
        The name of the column to plot.
    """
    plt.figure(figsize=(10, 4))
    plt.plot(df[col_name], marker='o', linestyle='-', linewidth=1)
    plt.title(f"{col_name} over index")
    plt.xlabel("Index")
    plt.ylabel(col_name)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Creating "df_mf_ib97sx_nn_s"

 ### This segment imports the data from the Excel filE .../data/ib97sx to dataframe with a list the name of all the files loaded

In [143]:
# loads the new from the 970
df_ib97sx_raw_uf = universal_import(
    folder_path="/home/ratlabs/JL_2/data/ib97sx",
    pattern="*"
)

✅ Loaded sx_20260810075104.csv with ISO-8859-1
✅ Loaded sx_20260821060645.csv with ISO-8859-1
✅ Loaded sx_20260822060650.csv with ISO-8859-1
✅ Loaded sx_20260824091237.csv with ISO-8859-1
✅ Loaded sx_20260829080159.csv with ISO-8859-1
✅ Loaded sx_20260806070510.csv with ISO-8859-1
✅ Loaded sx_20260831071250.csv with ISO-8859-1
✅ Loaded sx_20260827092659.csv with ISO-8859-1
✅ Loaded sx_20260816064922.csv with ISO-8859-1
✅ Loaded sx_20260809075417.csv with ISO-8859-1
✅ Loaded sx_20260901082536.csv with ISO-8859-1
✅ Loaded sx_20260825075451.csv with ISO-8859-1
✅ Loaded sx_20260820074538.csv with ISO-8859-1
✅ Loaded sx_20260818080327.csv with ISO-8859-1
✅ Loaded sx_20260807072021.csv with ISO-8859-1
✅ Loaded sx_20260817070339.csv with ISO-8859-1
✅ Loaded sx_20260815082452.csv with ISO-8859-1
✅ Loaded sx_20260826063355.csv with ISO-8859-1
✅ Loaded sx_20260830070230.csv with ISO-8859-1
✅ Loaded sx_20260812074925.csv with ISO-8859-1
✅ Loaded sx_20260813075018.csv with ISO-8859-1
✅ Loaded sx_2

In [144]:
# Data frame Unfiltered 
# verify 
df_ib97sx_raw_uf         

,1. Name,2. ID,3. Height,4. Date of Birth,5. Gender,6. Age,7. Mobile Number,8. Phone Number,9. Zip Code,10. Address,...,244. 50kHz-Whole Body Phase Angle_Z score,245. TBW/WT_T Score,246. TBW/WT_Z Score,247. SMI(SMM/Wt)_T score,248. SMI(SMM/Wt)_Z score,249. ECM/BCM_T Score,250. ECM/BCM Z Score,Unnamed: 250,source_file,encoding_used
0,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,0.2,-0.8,0.0,-0.9,0.0,2.2,0.1,NaN,sx_20260810075104.csv,ISO-8859-1
1,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.1,-0.6,0.2,-0.7,0.3,2.4,0.3,NaN,sx_20260821060645.csv,ISO-8859-1
2,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.3,-0.4,0.4,-0.6,0.4,2.0,0.0,NaN,sx_20260822060650.csv,ISO-8859-1
3,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.3,-0.3,0.5,-0.5,0.5,2.5,0.4,NaN,sx_20260824091237.csv,ISO-8859-1
4,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.3,-0.6,0.2,-0.8,0.2,2.1,0.0,NaN,sx_20260829080159.csv,ISO-8859-1
5,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.1,-0.8,0.0,-0.9,0.0,2.3,0.2,NaN,sx_20260806070510.csv,ISO-8859-1
6,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.3,-0.6,0.2,-0.8,0.2,2.0,0.0,NaN,sx_20260831071250.csv,ISO-8859-1
7,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.1,-0.6,0.2,-0.7,0.3,2.4,0.3,NaN,sx_20260827092659.csv,ISO-8859-1
8,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.3,-0.6,0.2,-0.8,0.2,2.3,0.3,NaN,sx_20260816064922.csv,ISO-8859-1
9,<sx>,sx,5ft 06.0in,1948.11.04.,F,77.0,-,-,-,-,...,-0.1,-0.8,-0.1,-1.0,0.0,2.0,0.0,NaN,sx_20260809075417.csv,ISO-8859-1


### df_ib97sx_raw_uf  Columns

In [145]:
#This is the list of df_ib97sx_raw columns
for col in df_ib97sx_raw_uf.columns:
    print(repr(col))

'1. Name'
'2. ID'
'3. Height'
'4. Date of Birth'
'5. Gender'
'6. Age'
'7. Mobile Number'
'8. Phone Number'
'9. Zip Code'
'10. Address'
'11. E-mail'
'12. Date of Registration'
'13. Memo'
'14. Test Date / Time'
'15. Weight'
'16. Lower Limit (Weight Normal Range)'
'17. Upper Limit (Weight Normal Range)'
'18. TBW (Total Body Water)'
'19. Lower Limit (TBW Normal Range)'
'20. Upper Limit (TBW Normal Range)'
'21. ICW (Intracellular Water)'
'22. Lower Limit (ICW Normal Range)'
'23. Upper Limit (ICW Normal Range)'
'24. ECW (Extracellular Water)'
'25. Lower Limit (ECW Normal Range)'
'26. Upper Limit (ECW Normal Range)'
'27. Protein'
'28. Lower Limit (Protein Normal Range)'
'29. Upper Limit (Protein Normal Range)'
'30. Minerals'
'31. Lower Limit (Minerals Normal Range)'
'32. Upper Limit (Minerals Normal Range)'
'33. DLM (Dry Lean Mass)'
'34. BFM (Body Fat Mass)'
'35. Lower Limit (BFM Normal Range)'
'36. Upper Limit (BFM Normal Range)'
'37. FFM (Fat Free Mass)'
'38. SMM (Skeletal Muscle Mass)'
'39

### Filtered rows BY the "IDcol" for the "ID1" . "ID2 " to choose between ib970 and ib 770

In [146]:
IDcol = "2. ID"
ID1 ="091725-1"
df_ib97sx_raw_uf[IDcol] = df_ib97sx_raw_uf[IDcol].astype(str)
df_ib97sx_raw = df_ib97sx_raw_uf[df_ib97sx_raw_uf[IDcol].isin([ID1])]


In [147]:
# verify df_ib97sx_raw['15. Weight']

### This segment strips off the col names FROM numbers ATTACHED BY iNBODY and produces "df_m_ib97sx_nn" and demonstrates slicing

In [148]:
# this Strips the numbers and spaces off column names and verifies the numbers are removed.
df_ib97sx_nn = strip_numbers_from_columns(df_ib97sx_raw)
#verify print(list(df_ib97sx_nn.columns))

In [149]:
# print(df_ib97sx_nn.columns.tolist())


In [150]:
#df_ib97sx_nn.columns[df_ib97sx_nn.columns.duplicated()]


In [151]:
df_ib97sx_nn.columns = (
    df_ib97sx_nn.columns
    .str.strip()
    .str.replace('\u200b','', regex=True)   # remove zero‑width chars
)


In [152]:
df_ib97sx_nn.columns.tolist()


['Name',
 'ID',
 'Height',
 'Date of Birth',
 'Gender',
 'Age',
 'Mobile Number',
 'Phone Number',
 'Zip Code',
 'Address',
 'E-mail',
 'Date of Registration',
 'Memo',
 'Test Date / Time',
 'Weight',
 'Lower Limit (Weight Normal Range)',
 'Upper Limit (Weight Normal Range)',
 'TBW (Total Body Water)',
 'Lower Limit (TBW Normal Range)',
 'Upper Limit (TBW Normal Range)',
 'ICW (Intracellular Water)',
 'Lower Limit (ICW Normal Range)',
 'Upper Limit (ICW Normal Range)',
 'ECW (Extracellular Water)',
 'Lower Limit (ECW Normal Range)',
 'Upper Limit (ECW Normal Range)',
 'Protein',
 'Lower Limit (Protein Normal Range)',
 'Upper Limit (Protein Normal Range)',
 'Minerals',
 'Lower Limit (Minerals Normal Range)',
 'Upper Limit (Minerals Normal Range)',
 'DLM (Dry Lean Mass)',
 'BFM (Body Fat Mass)',
 'Lower Limit (BFM Normal Range)',
 'Upper Limit (BFM Normal Range)',
 'FFM (Fat Free Mass)',
 'SMM (Skeletal Muscle Mass)',
 'Lower Limit (SMM Normal Range)',
 'Upper Limit (SMM Normal Range)',


In [153]:
df_ib97sx_nn.columns = (
    df_ib97sx_nn.columns
    .str.strip()
    .str.replace('\u200b','', regex=True)
    .to_series()
    .pipe(lambda s: s + s.groupby(s).cumcount().replace(0,'').astype(str))
    .values
)


In [154]:
# verify 
df_ib97sx_nn[['ID', 'Test Date / Time', 'ECW/TBW']]
 # ,'15. Weight'

,ID,Test Date / Time,ECW/TBW


In [155]:
df_ib97sx_nn_s = df_ib97sx_nn.sort_values(by="Test Date / Time").reset_index(drop=True)

In [156]:
# verify 
df_ib97sx_nn_s[['ID','Test Date / Time','ECW/TBW']]

,ID,Test Date / Time,ECW/TBW


### This segment adds **media_cols** and fills them from data in the results **"df_m_ib97sx_nn"**

In [157]:
df_m_ib97sx_nn = prepend_empty_columns(df_ib97sx_nn, media_lst)
# verify 
df_m_ib97sx_nn

,timestamp,dtv,ib_id,cls,cmmnts,Name,ID,Height,Date of Birth,Gender,...,50kHz-Whole Body Phase Angle_Z score,TBW/WT_T Score,TBW/WT_Z Score,SMI(SMM/Wt)_T score,SMI(SMM/Wt)_Z score,ECM/BCM_T Score,ECM/BCM Z Score,Unnamed,source_file,encoding_used


In [160]:
# fill_ib_media_cols(df_mf_ib97sx_nn)

df_mf_ib97sx_nn = fill_ib_media_cols(df_m_ib97sx_nn)
# verify df_mf_ib97sx_nn

TypeError: unsupported operand type(s) for -: 'numpy.ndarray' and 'Timestamp'

### This segment eliminates "COL" duplicates in **df_mf_ib97sx_nn**

In [ ]:
######## df_mf_ib97sx_nn = drop_duplicate_columns(df_mf_ib97sx_nn)
print ("df_mf_ib97sx_nn , No numbers, no duplicates OK")

### This segment sorts "df_mf_ib97sx_nn" to get "df_mf_ib97sx_nn_s"

In [ ]:
df_mf_ib97sx_nn_s = sort_by_timestamp(df_mf_ib97sx_nn )
# verify df_mf_ib97sx_nn_s

### This segment eliminates "ROW" duplicates based on **"test_time" in **"df_mf_ib97sx_nn"** note: "mf" means media added and filled]

In [ ]:
df_mf_ib97sx_nn_s = df_mf_ib97sx_nn_s.drop_duplicates(subset="timestamp", keep="first")
print("No timestamp duplicates **df_mf_ib97sx_nn** detected")

In [ ]:
# verify list(df_mf_ib97sx_nn_s.columns)

# Set the "plt_lst" [the list cols to be plotted BY COL HEAD NAME]
1. **"df_mf_ib97sx_nn_s"** is the source data 

In [ ]:
plt_lst = ["ECW/TBW",
           "BMR (Basal Metabolic Rate)",
           
           "SMM (Skeletal Muscle Mass)",
           "VFA (Visceral Fat Area)"]


### filtering a errent row

You can remove **row 951** from your DataFrame in Jupyter Lab using standard pandas operations. The key is whether you want to remove it **by index label** or **by position** — those are different things in pandas.

---

## ✅ **Fast Answer**
If **951 is the index value**:

```python
df_mf_ib97sx_nn_s = df_mf_ib97sx_nn_s.drop(951)
```

If **951 is the row number (0‑based position)**:

```python
df_mf_ib97sx_nn_s = df_mf_ib97sx_nn_s.drop(df_mf_ib97sx_nn_s.index[951])
```

---

## 🧠 **How to tell which one you need**
Print the index:

```python
df_mf_ib97sx_nn_s.index
```

If you see something like:

```
Int64Index([0, 1, 2, ..., 951, ...], dtype='int64')
```

→ Use the **index version**.

If the index is something else (dates, strings, etc.) but you know the *position* is 951:

→ Use the **position version**.

---

## 🧹 Optional: Remove the row *in place*
If you want to modify the DataFrame without reassigning:

```python
df_mf_ib97sx_nn_s.drop(df_mf_ib97sx_nn_s.index[951], inplace=True)
```

---

## 🔍 Verify it’s gone
```python
df_mf_ib97sx_nn_s.loc[951]      # if index-based
df_mf_ib97sx_nn_s.iloc[951]     # if position-based
```

You should get a KeyError or see that the row no longer exists.

---

If you want, I can help you inspect the DataFrame to confirm whether 951 is an index or a position — just tell me what `df_mf_ib97sx_nn_s.index` prints.

You can filter those rows cleanly with pandas using a boolean condition on the **ecw/tbw** column.

---

## ✅ **Concise Answer**
```python
df_mf_ib97sx_nn_s[df_mf_ib97sx_nn_s["ecw/tbw"] < 0.389]
```

This returns **all rows** where the value in **ecw/tbw** is less than **0.389**.

---

## 🧠 **If you want to *remove* those rows instead**
```python
df_mf_ib97sx_nn_s = df_mf_ib97sx_nn_s[df_mf_ib97sx_nn_s["ecw/tbw"] >= 0.389]
```

This keeps only rows **greater than or equal to 0.389**.

---

## 🔍 **If you want to inspect how many rows match**
```python
(df_mf_ib97sx_nn_s["ecw/tbw"] < 0.389).sum()
```

---

## 📌 **If the column name has special characters**
Pandas handles it fine as long as you use quotes:

```python
df_mf_ib97sx_nn_s[df_mf_ib97sx_nn_s["ecw/tbw"] < 0.389]
```

But if you prefer attribute-style access, you can rename it:

```python
df_mf_ib97sx_nn_s = df_mf_ib97sx_nn_s.rename(columns={"ecw/tbw": "ecw_tbw"})
```

Then:

```python
df_mf_ib97sx_nn_s[df_mf_ib97sx_nn_s.ecw_tbw < 0.389]
```

---

If you want, I can help you build a reusable function to scan for outliers or threshold violations across any column — just tell me whether you want a simple filter, a diagnostic printout, or an automated cleaning step.

In [ ]:
#Test if the number is index
#df_mf_ib97sx_nn_s.index


In [ ]:
# delete row "951" 
#df_mf_ib97sx_nn_s = df_mf_ib97sx_nn_s.drop(951)


In [ ]:
# verify df_mf_ib97sx_nn_s 

In [ ]:
# Technique for Splicing"timestamp" and "a row" from "df_mf_ib97sx_nn_s"
# verify df_mf_ib97sx_nn_s[["timestamp","ID","ECW/TBW",'BMR (Basal Metabolic Rate)',"SMM (Skeletal Muscle Mass)",'VFA (Visceral Fat Area)']]        

In [ ]:
# Selects the cols to be plotted inserting the media columns before the printed columns
plt_lst = ["ECW/TBW", 'BMR (Basal Metabolic Rate)',"SMM (Skeletal Muscle Mass)",
           'VFA (Visceral Fat Area)','50kHz-Whole Body Phase Angle','Test Date / Time1']
# verify df_mf_ib97sx_nn_s[["ID","timestamp",'ib_id']+plt_lst]

In [ ]:
# This filters out only the morning values of chosen data Col
############df_mf_ib97sx_nn_s_mrn = filter_by_value(df_mf_ib97sx_nn_s, 'ib_id', "mrn")


In [ ]:
# verify df_mf_ib97sx_nn_s_mrn

In [ ]:
# This allows the full 77 data frame with columns to sort out for 7797 morning evening ETC
write_df_to_pickle(df_mf_ib97sx_nn_s, "df_mf_ib97sx_nn_s.pkl")
print("df_mf_ib97sx_nn_s written to pickle")


# Normalize and Subract 1 on "df_mf_ib97sx_nn_s[plt_lst]" col by col

In [ ]:
# df_mf_ib97sx_nn_s_n0 = df_mf_ib97sx_nn_s.copy() 
#############df_mf_ib97sx_nn_s_mrn_n0 = df_mf_ib97sx_nn_s_mrn.copy() 
# VERIFY0df_mf_ib97sx_nn_s_mrn_n0['ECW/TBW']

In [ ]:
# VERIFY
df_mf_ib97sx_nn_s_mrn_n0['50kHz-Whole Body Phase Angle']

In [ ]:
plt_col = "ECW/TBW"


In [ ]:
df_mf_ib97sx_nn_s_mrn_n0[plt_col]  =   scale_mean_to_one(df_mf_ib97sx_nn_s_mrn_n0[plt_col])-1
# verify df_mf_ib97sx_nn_s_mrn_n0[plt_col]

In [ ]:
# plot_column(df_mf_ib97sx_nn_s_mrn_n0, plt_col)

In [ ]:
plt_lst = ['Weight', "ECW/TBW", 'BMR (Basal Metabolic Rate)',"SMM (Skeletal Muscle Mass)", 'VFA (Visceral Fat Area)',
          '50kHz-Whole Body Phase Angle', 'Test Date / Time1']

In [ ]:
for plt_col in plt_lst:
    # Print title of eacgh graph
    print("Now plotting:", plt_col)
    # Calc the  normalized dirivaative of each col
    df_mf_ib97sx_nn_s_mrn_n0[plt_col]  =   scale_mean_to_one(df_mf_ib97sx_nn_s_mrn_n0[plt_col])-1
    plot_column(df_mf_ib97sx_nn_s_mrn_n0, plt_col)

## ***JL_2 Repo*** ib_97 sys ___ 1st example useing a root system templates

# WORKING cell 

In [ ]:
# verify df_mf_ib97sx_nn_s

In [ ]:
# Misc Loadi3ng working file
filename = "df_mf_ib97sx_nn_s.pkl"

dfxxx = load_df_from_pickle(filename)

# verify dfxxx



In [ ]:
# verify 
print("df_ib97sx_raw  loaded  OK")